### ⭐ Why timeit is better than manual timing
Note: `timeit` is meant for **external benchmarking**, not for decorators.

When you do:
```py
import time

start = time.time()
fib(30)
print(time.time() - start)
```
You measure:
- interpreter warm‑up
- cache effects
- background OS noise
- one single run (which is statistically meaningless)

`timeit` solves all of this by:
- running your code many times
- disabling garbage collection
- isolating the code in a clean environment
- reporting the best or average time

This gives stable, trustworthy results.

### ✅ The simplest way: using `timeit.timeit()`

In [4]:
import timeit


def fib_iterative(number):
    a, b = 0, 1
    for _ in range(number):
        a, b = b, a + b
    return a

print(
    timeit.timeit(
        "fib_iterative(30)",
        setup="""from __main__ import fib_iterative""",
        number=100000
    )
)

0.06851660000029369


Explanation:
- setup runs once (imports your function)
- stmt runs number times
- returns total time for all runs

---

### ⭐ Benchmarking your Fibonacci functions

In [5]:
def fib_iterative(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

In [6]:
from functools import lru_cache

@lru_cache(None)
def fib_cached(n):
    if n <= 1:
        return n
    return fib_cached(n-1) + fib_cached(n-2)

In [7]:
# Benchmark iterative:
time_iter = timeit.timeit(
    "fib_iterative(30)",
    setup="from __main__ import fib_iterative",
    number=100000
)

print("Iterative:", time_iter)

Iterative: 0.07447980000142707


In [8]:
# Benchmark recursive + memo:
time_memo = timeit.timeit(
    "fib_cached(30)",
    setup="from __main__ import fib_cached",
    number=100000
)

print("Memoized recursive:", time_memo)

Memoized recursive: 0.004371099999843864


### ⭐ Even cleaner: use `timeit.repeat()` for multiple runs
This gives you multiple samples:

In [9]:
times = timeit.repeat(
    "fib_iterative(30)",
    setup="from __main__ import fib_iterative",
    number=100000,
    repeat=5
)

print(times)
print("Best:", min(times))

[0.07248629999958212, 0.06285270000080345, 0.06444659999760916, 0.07059339999977965, 0.09005880000040634]
Best: 0.06285270000080345


This is the gold standard for benchmarking.

---

### ⭐ Cleanest of all: use `timeit` from the command line
This avoids Python startup overhead entirely.

Example:
```shell
python -m timeit -s "from fib import fib_iterative" "fib_iterative(30)"
```
Or:
```shell
python -m timeit -s "from fib import fib_cached" "fib_cached(30)"
```
This is what Python core developers use.

---

### ⭐ Summary (what you should remember)
- Use `timeit.timeit()` for simple benchmarks
- Use `timeit.repeat()` for more reliable results
- Use `python -m timeit` for the cleanest environment
- Never benchmark recursive Fibonacci without memoization
- Never benchmark with manual `time.time()` unless you're debugging

### Example

In [11]:
def memoize(func):
    cache = {}
    def wrapper(*args):
        if args in cache:
            return cache[args]
        result = func(*args)
        cache[args] = result
        return result
    return wrapper


@memoize
def fibonacci(n):
    if n <= 1:
        return n
    return fibonacci(n-1) + fibonacci(n-2)

#### Measuring only the cached lookup
```shell
py -m timeit -s "from main import fibonacci" "fibonacci(40)"

5000000 loops, best of 5: 91 nsec per loop
```
- `5000000 loops`: `timeit` automatically chose to run your statement 5,000,000 times to get a stable measurement
- `best of 5: 91 nsec per loop`: `timeit` ran 5 separate timing rounds and picked the fastest one.

Each call to `fibonacci(40)` (after caching) took 91 nanoseconds — basically just a dictionary lookup.

- `6.09 usec per loop`: moderately fast — much slower than a cached lookup (nanoseconds), but still very quick in absolute terms.

---

Why does timeit show 5,000,000 loops and 91 nanoseconds per loop?

Because with memoization:
```python
@memoize
def fibonacci(n):
    ...
```
**all Fibonacci values are computed exactly once**.

So when you run:
```shell
py -m timeit -s "from main import fibonacci" "fibonacci(40)"
```
Here’s what happens:

✔ Step 1 — setup runs once
- This imports your memoized function.

✔ Step 2 — stmt runs 5,000,000 times
- Each time it calls: `fibonacci(40)`

But because the result is cached:
- The first call computes the full recursion tree
- All subsequent calls return instantly from the cache

So the benchmark is measuring **only the cached lookup**, not the actual Fibonacci computation.

✔ A cached lookup is extremely fast
- A dictionary lookup in Python is ~50–100 nanoseconds.

Your result: `91 nsec per loop`

is exactly what we expect for: `cache[(40,)]  # dictionary lookup`

---

#### Why does it print `102334155` before the timing?
Because your function prints the result the first time it runs.

`timeit` executes the statement **once before timing** to ensure it’s valid.

So:
- First run → prints the Fibonacci number
- Then timeit runs the statement millions of times silently

This is normal.

---

#### Important: You did NOT benchmark Fibonacci computation
You benchmarked cache retrieval.

To benchmark the actual recursive computation, you must:

Option A — Clear the cache before each run
(But your custom decorator doesn’t support clearing.)

Option B — Use `lru_cache` and clear it manually

In [12]:
from functools import lru_cache

@lru_cache(None)
def fib(n):
    if n <= 1:
        return n
    return fib(n-1) + fib(n-2)

Then benchmark:
```shell
py -m timeit -s "from main import fib; fib.cache_clear()" "fib(40); fib.cache_clear()"
```
This forces recomputation every time.

Option C — Benchmark the naive recursive version

(Warning: it will be extremely slow for n=40.)

### What would the real timings look like?

1. Naive recursion ($O(2^n)$): `fib(40) → ~1 billion operations → several seconds`

2. Memoized recursion ($O(n)$): `fib(40) → ~40 operations → microseconds`

3. Cached lookup: `fib(40) → dictionary lookup → ~90 nanoseconds`

Your measurement corresponds to the third case.

---

### Summary of your results
- Your memoized Fibonacci works correctly
- `timeit` is measuring **cached lookups**, not computation
- 91 ns per call is exactly what a dictionary lookup costs
- The printed Fibonacci number is from the first warm‑up call
- Your benchmark is valid, but it’s measuring the wrong thing